(dkist:tutorial:coordinates)=

# Coordinates - A Quick Primer

This chapter will cover the basic usage of the coordinates framework in SunPy and Astropy, and related functionality, which is relevant to the `dkist` package.
There are many other parts of these packages which are useful when working with DKIST data, which you should explore in their respective documentation pages.
See [here for SunPy's documentation](https://docs.sunpy.org/) and [here for astropy's](https://docs.astropy.org/).

In this chapter you will:

- Convert values between different physical units
- Define spatial and spectral coordinates
- Convert between world and pixel coordinate systems

## Units

Astropy provides a subpackage <a href="https://docs.astropy.org/en/stable/units/ref_api.html#module-astropy.units" target="_blank" style="text-decoration: underline">`astropy.units`</a> which provides tools for associating physical units with numbers and arrays.
This lets you do mathematical operations on these arrays while propagating the units.

To get started we import `astropy.units`.
Because we are going to be using this a lot, we import it as `u`.


In [ ]:
import astropy.units as u

Now we can access various physical units defined in astropy, such as metres or kilograms:


In [ ]:
u.m

In [ ]:
u.kg

We can also attach a unit to a number to create a quantity:


In [ ]:
length = 10 * u.m
length

And use multiple quantities in a calculation:


In [ ]:
time = 30 * u.min
speed = length / time
speed

Using the `.to()` method on a `u.Quantity` object lets you convert a quantity to a different unit.


In [ ]:
speed.to(u.km/u.h)

### Equivalencies

Some conversions are not done by a conversion factor as between miles and kilometers – for example converting between wavelength and frequency:


In [ ]:
(656.281 * u.nm).to(u.Hz)  # Fails because they are not compatible

However we can make use of a spectral *equivalency* to indicate the link between the units:


In [ ]:
(656.281 * u.nm).to(u.Hz, equivalencies=u.spectral())

### Constants

The `astropy.constants` sub-package provides a set of physical constants which are compatible with the units/quantities framework:


In [ ]:
from astropy.constants import M_sun, c

In [ ]:
E = M_sun * c ** 2
E.to(u.J)

## Coordinates

The Astropy coordinates submodule <a href="https://docs.astropy.org/en/stable/coordinates/ref_api.html#module-astropy.coordinates" target="_blank" style="text-decoration: underline">`astropy.coordinates`</a> provides classes to represent physical coordinates with all their associated metadata, and transform them between different coordinate systems.
Currently, <a href="https://docs.astropy.org/en/stable/coordinates/ref_api.html#module-astropy.coordinates" target="_blank" style="text-decoration: underline">`astropy.coordinates`</a> supports:

- Spatial coordinates via <a href="https://docs.astropy.org/en/stable/api/astropy.coordinates.SkyCoord.html#astropy.coordinates.SkyCoord" target="_blank" style="text-decoration: underline">`astropy.coordinates.SkyCoord`</a>
- Spectral coordinates via <a href="https://docs.astropy.org/en/stable/api/astropy.coordinates.SpectralCoord.html#astropy.coordinates.SpectralCoord" target="_blank" style="text-decoration: underline">`astropy.coordinates.SpectralCoord`</a>
- Stokes profiles via <a href="https://docs.astropy.org/en/stable/api/astropy.coordinates.StokesCoord.html#astropy.coordinates.StokesCoord" target="_blank" style="text-decoration: underline">`astropy.coordinates.StokesCoord`</a> (introduced in astropy 7.0)

### Spatial Coordinates

SunPy provides extensions to the Astropy coordinate system to represent common solar physics frames.
So to use the sunpy coordinates we have to first import <a href="https://docs.sunpy.org/en/stable/reference/coordinates/index.html#module-sunpy.coordinates" target="_blank" style="text-decoration: underline">`sunpy.coordinates`</a> which registers them with astropy.


In [ ]:
import sunpy.coordinates
from astropy.coordinates import SkyCoord

We can now create a `SkyCoord` object representing a point on the Sun:


In [ ]:
SkyCoord(10*u.arcsec, 20*u.arcsec, frame="helioprojective")

This is the most minimal version of this coordinate frame, however, it isn't very useful as we haven't provided enough information to be able to transform it to other frames.
This is because helioprojective is an observer centred coordinate frame, so we need to know where in inertial space the observer is.
One way of doing this is to say the observer is on Earth at a specific time:


In [ ]:
hpc1 = SkyCoord(10*u.arcsec, 20*u.arcsec, frame="helioprojective",
                obstime="2023-05-21T04:00:00", observer="earth")
hpc1

This coordinate can now be converted to other frames, such as heliographic coordinates:


In [ ]:
hpc1.transform_to("heliographic_stonyhurst")

There are few things to notice about the difference between these two `SkyCoord` objects:

1. The default representation of the latitude and longitude is now in degrees as is conventional.
1. The heliographic frame is three dimensional (it has a radius), when the input frame was not. This is because the distance from the observer was calculated using the `rsun` attribute.
1. The `obstime` and `rsun` attributes are still present, but the `observer` attribute isn't. This is because heliographic coordinates are not observer dependent.
1. The `obstime` attribute is still important to transform to other frames, as the heliographic frame needs to know the location of Earth.

### Spectral Coordinates

<a href="https://docs.astropy.org/en/stable/api/astropy.coordinates.SpectralCoord.html#astropy.coordinates.SpectralCoord" target="_blank" style="text-decoration: underline">`astropy.coordinates.SpectralCoord`</a> is a `Quantity`-like object which also holds information about the observer and target coordinates and relative velocities.

```{note}
Use of `SpectralCoord` with solar data is still experimental so not all features may work, or be accurate.
```


In [ ]:
from astropy.coordinates import SpectralCoord
from sunpy.coordinates import get_earth

`SpectralCoord` does not automatically make the HPC coordinate 3D, but wants the distance, so we do it explicitally:


In [ ]:
hpc2 = hpc1.make_3d()

Now we can construct our spectral coordinate with both a target and an observer


In [ ]:
spc = SpectralCoord(586.3 * u.nm, target=hpc2, observer=get_earth(time=hpc2.obstime))
spc

(If you're viewing this document in Jupyter notebook form you may have to work around a [bug in astropy](https://github.com/astropy/astropy/issues/14758) to display `spc` properly):


In [ ]:
print(repr(spc))

## World Coordinate System

One of the other core components of the ecosystem provided by Astropy is the <a href="https://docs.astropy.org/en/stable/wcs/reference_api.html#module-astropy.wcs" target="_blank" style="text-decoration: underline">`astropy.wcs`</a> package which provides tools for mapping pixel to world coordinates and world to pixel.
When loading a FITS file with complete (and standard compliant) WCS metadata we can create an `astropy.wcs.WCS` object.
For this example we will use a sample VISP header distributed with the `dkist` package.


In [ ]:
import sunpy.coordinates

To read this FITS file we will use <a href="https://docs.astropy.org/en/stable/io/fits/index.html#module-astropy.io.fits" target="_blank" style="text-decoration: underline">`astropy.io.fits`</a> (you can also use `sunpy` for this).


In [ ]:
from dkist.data.sample import VISP_HEADER

Using this header we can create a `astropy.wcs.WCS` object:


In [ ]:
from astropy.wcs import WCS

wcs = WCS(VISP_HEADER)
wcs

This WCS object allows us to convert between pixel and world coordinates, for example:


In [ ]:
wcs.pixel_to_world(1024, 400, 1)

This call returns a <a href="https://docs.astropy.org/en/stable/api/astropy.coordinates.SkyCoord.html#astropy.coordinates.SkyCoord" target="_blank" style="text-decoration: underline">`astropy.coordinates.SkyCoord`</a> object (which needs sunpy to be imported), we will come onto this more later.

We can also convert between pixel and plain numbers:


In [ ]:
wcs.pixel_to_world_values(1024, 400, 1)

The units for these values are given by:


In [ ]:
wcs.world_axis_units

The WCS also has information about what the world axes represent:


In [ ]:
wcs.world_axis_physical_types

We can also inspect the correlation between the world axes and pixel axes:


In [ ]:
wcs.axis_correlation_matrix

This correlation matrix has the world dimensions as rows, and the pixel dimensions as columns.
Here we have a 2D image, with two pixel and two world axes where both are coupled together.
This means that to calculate either latitude or longitude you need both pixel coordinates.
